In [ ]:
import sklearn
import pandas
from sklearn import datasets

print("sklearn:", sklearn.__version__)
print("pandas:", pandas.__version__)

sklearn: 1.9.1
pandas: 3.0.6


In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [3]:
# ---------------------------------------------
# 1. 데이터 불러오기
# ---------------------------------------------
datasets = load_breast_cancer()
X = pd.DataFrame(datasets.data, columns=datasets.feature_names)
y = pd.Series(datasets.target, name="target") # 0: 악성, 1:양성

print("X 형태 (샘플 수, 특성 수):", X.shape)
print("클래스 비율:")
print(y.value_counts(normalize=True))

X 형태 (샘플 수, 특성 수): (569, 30)
클래스 비율:
target
1    0.627417
0    0.372583
Name: proportion, dtype: float64


In [5]:
# ---------------------------------------------
# 2. 데이터 분할 (계층 분할 + 재현성 고정)
# ---------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("훈련 데이터:", X_train.shape)
print("테스트 데이터:", X_test.shape)

훈련 데이터: (455, 30)
테스트 데이터: (114, 30)


In [8]:
# ---------------------------------------------
# 3~6. 트리 깊이별로 과적합 확인
# ---------------------------------------------
depth_list = [1,2,3,5,10,None] # None : 깊이 제한 없음
results = []

for depth in depth_list:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)

    #학습
    model.fit(X_train, y_train)

    #훈련 점수
    train_pred = model.predict(X_train)
    train_acc = accuracy_score(y_train, train_pred)

    #교차 검증
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    results.append({
        "max_depth": str(depth),
        "train_acc": round(train_acc, 3),
        "cv_mean": round(cv_mean, 3),
        "cv_std": round(cv_std, 3),
        "gap": round(train_acc - cv_mean, 3)
    })

result_df = pd.DataFrame(results)
print(result_df)

  max_depth  train_acc  cv_mean  cv_std    gap
0         1      0.923    0.897   0.019  0.026
1         2      0.958    0.919   0.019  0.040
2         3      0.976    0.925   0.024  0.051
3         5      0.993    0.932   0.021  0.062
4        10      1.000    0.910   0.019  0.090
5      None      1.000    0.910   0.019  0.090


In [9]:
# ---------------------------------------------
# 7. 교차검증 기준으로 최적 깊이 선택 → 테스트는 딱 한 번
# ---------------------------------------------
best_row = result_df.loc[result_df["cv_mean"].idxmax()]
best_depth_text = best_row["max_depth"]

if best_depth_text == "None":
    best_depth = None
else:
    best_depth = int(best_depth_text)

final_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
final_model.fit(X_train, y_train)

test_pred = final_model.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)


print("선택된 깊이:", best_depth)
print("최종 테스트 정확도:", round(test_acc, 3))

선택된 깊이: 5
최종 테스트 정확도: 0.921
